# A9 · StepFun Step-5-Preview (sliding + DSA · MoE) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A9_step5.ipynb)

원본: HF `stepfun-ai/Step-5-Preview-BF16`의 `inference/model.py` + `kernel.py` (커밋 `7734b7f` 고정).
transformers main에는 아직 step5가 없어서 StepFun 공식 추론 코드를 받아 씁니다.
이 코드는 Triton 커널을 쓰므로 **커널을 부르는 함수만** 같은 수식의 fp32 PyTorch 구현으로 바꿔 끼웁니다
(fp8/e4m3 반올림은 항등, bf16 저장은 fp32. 목록은 `miniarch/verify/stages/a9_step5.py` 맨 위 설명).
vision encoder와 MTP 층은 비교하지 않습니다 (텍스트 backbone만).

내 구현: `miniarch/models/step5.py`. 가중치 이름은 실제 체크포인트 키(`model.layers.N...`)와 같습니다.

부품
- sliding 3 : full 1 하이브리드 (92층). sliding 층은 window 512, full 층은 DSA(sparse) attention
- 층마다 다른 RoPE: sliding은 head 전체 회전 · theta 1e4, full은 앞 1/3만 회전 · theta 1e7 + llama3 scaling(full 층만)
- GQA(q 64 / kv 4), q·k에 zero-centered RMSNorm `x·(1+w)`, 출력에 head별 sigmoid gate
- DSA indexer: 토큰 8개 region을 채널별 softmax(z) 가중평균으로 요약하고, kv group마다 `Σ w·ReLU(q·요약)`으로
  이전 region top-512 + 현재 region만 봄
- MoE: expert 352개 중 8개 (fp32 sigmoid + 선택용 bias, 정규화 × 3.0) + shared expert, 앞 3층은 dense.
  SwiGLU는 gate·up 모두 clamp(±7). residual은 fp32로 누적

축소 config (모든 경로가 한 번씩 쓰이도록 고름)
- 5층: [sliding+dense, full(DSA)+dense, sliding+MoE, sliding+MoE, full(DSA)+MoE]
- head_dim 24: full 층 회전 비율 1/3 → 8채널, 0.3 → 7 → 짝수로 6채널 (반올림 경로). theta 1e4 / 1e6
- llama3 scaling: original 512, factor 4, 대역 1~4 → 회전 주파수가 그대로 / 보간 / 1/4 세 구간에 모두 걸림
- DSA: region 4토큰, top-3, indexer head 4개(kv group 2 × 2), proxy 32, indexer RoPE 8채널
- 시퀀스 45 / 38 (오른쪽 padding): region이 최대 11개라 top-3이 실제로 고르고 버림, 덜 찬 마지막 region,
  요청마다 다른 길이(packed 경계), window 12 밖 토큰까지 지나감
- SwiGLU limit을 0.6~1.0으로 작게 둬서 clamp가 실제로 걸림. expert 8개 중 3개
- 원본이 `torch.empty`/0으로 만드는 가중치는 0이 아닌 값으로 채웁니다.
- 층별(attention / routed MoE / shared / dense MLP / 층 출력)과 최종 norm, 전 위치 logits를 비교합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 학습 동작 확인 → 6. 단계 전용 자체 점검 → 7. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A9")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('stepfun-ai/Step-5-Preview-BF16', '^model\\.layers\\.(2|3)\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

## 단계 전용 자체 점검

### 커널 대체 확인

원본 비교의 기준값이 커널 대체에 달려 있으므로, 대체한 region 선택(`prefill_sparse_meta`)이 정의
(질의 region 이전 region 중 점수 top-k, 동점은 번호 작은 쪽 + 현재 region)와 같은지,
`sparse_attention_prefill`이 고른 토큰에 대한 softmax와 같은지 따로 확인합니다. 0 동점을 일부러 많이 넣습니다.
GPU에서 원본 Triton 커널이 돌아가면 (GPU 세대에 따라 안 될 수 있음) 원본과도 직접 비교합니다.

In [ ]:
V.set_deterministic(0)
rows_, W_, R_, topk_ = 40, 9, 4, 3
pos_ = torch.randint(0, W_ * R_ + R_, (rows_,), dtype=torch.int32)
sc_ = (torch.randint(-2, 3, (rows_, W_)).float() * 0.5).clamp_min(0)       # ReLU 처럼 0 동점이 많음
table_ = torch.arange(8, dtype=torch.int32).view(1, -1)                     # 모든 요청이 쓰는 page 표 하나
packed_, cnt_ = S.prefill_sparse_meta(sc_, pos_, table_, None, topk=topk_, region_size=R_, regions_per_page=4)
ok_ = True
for r in range(rows_):
    p = int(pos_[r]); vis = min(p // R_, W_)
    want = sorted(sorted(range(vis), key=lambda j: (-float(sc_[r, j]), j))[:topk_]) + [p // R_]
    got = [x & 0xFFFFFF for x in packed_[r, :int(cnt_[r])].tolist() if x >= 0]
    ok_ = ok_ and got == want
report("region 선택 대체 == 정의 (동점 포함)", ok_, 0.0)

G_, H_, D_ = 2, 4, 16
Tq_ = rows_ // G_
q_, kc_, vc_ = torch.randn(Tq_, H_, D_), torch.randn(64, G_, D_), torch.randn(64, G_, D_)
o1, _ = S.sparse_attention_prefill(q_, kc_, vc_, packed_, cnt_, num_kv_groups=G_, region_size=R_, softmax_scale=0.3)
o2 = torch.zeros_like(q_)
for g in range(G_):
    for t in range(Tq_):
        toks = []
        for x in packed_[g * Tq_ + t, :int(cnt_[g * Tq_ + t])].tolist():
            if x >= 0:
                toks += [(x & 0xFFFFFF) * R_ + o for o in range(x >> 24)]
        idx = torch.tensor(toks)
        for h in range(g * (H_ // G_), (g + 1) * (H_ // G_)):
            o2[t, h] = ((kc_[idx, g] @ q_[t, h]) * 0.3).softmax(0) @ vc_[idx, g]
err = (o1 - o2).abs().max().item()
report("sparse_attention_prefill 대체 == 정의", err < 1e-5, err)

S.load_reference()
O_ = S._LOADED["orig"]
if dev == "cuda":
    try:
        p2, c2 = O_["prefill_sparse_meta"](sc_.to(dev), pos_.to(dev), table_.to(dev), None,
                                            topk=topk_, region_size=R_, regions_per_page=4)
        same = torch.equal(p2.cpu(), packed_) and torch.equal(c2.cpu(), cnt_)
        report("region 선택: 원본 Triton == 대체", same, 0.0)
        o3, _ = O_["sparse_attention_prefill"](q_.to(dev), kc_.to(dev), vc_.to(dev), p2, c2,
                                               num_kv_groups=G_, region_size=R_, softmax_scale=0.3)
        err = (o3.cpu() - o1).abs().max().item()
        report("sparse attention: 원본 Triton == 대체 (tf32 허용)", err < 5e-3, err)
    except Exception as e:
        print("⚠️ 원본 Triton 커널을 이 GPU/버전에서 실행하지 못해 건너뜀:", type(e).__name__, str(e)[:200])

### DSA 선택 · attention map 확인 (내 구현)

- full(DSA) 층: 질의 위치 p 마다 고른 과거 region 수 = min(top-k, p // region) 이어야 합니다 (kv group 마다).
- sliding 층: 미래 칸과 window 밖 칸의 확률은 정확히 0.
- 모든 층: 확률 행 합 = 1.

In [ ]:
from miniarch.models import step5 as S5
ref, inp = S.ref()
m = S5.verify_pair(ref)["mine"].to(dev).eval()
c = m.config
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "select_probe"))]
x = inp["input_ids"].to(dev)
T_ = x.shape[1]
pos = torch.arange(T_, device=dev)
with V.capture(m, names) as st, torch.no_grad():
    m(x)
for i, layer in enumerate(m.model.layers):
    a = layer.self_attn
    p = st[f"model.layers.{i}.self_attn.attn_probs"]
    err = (p.sum(-1) - 1).abs().max().item()
    if a.sparse:
        sel = st[f"model.layers.{i}.self_attn.select_probe"]
        n_sel = sel.sum(-1).float()
        want = (pos // c.region).clamp_max(c.index_topk).float()
        seen = (p > 0).sum(-1).float().mean().item()
        print(f"layer {i} DSA  rope {a.span}채널  질의당 고른 region 평균 {n_sel.mean():.2f}개  평균 본 토큰 {seen:.1f}")
        report(f"layer {i} 고른 region 수 = min(top-k, p//R)", bool((n_sel == want).all()), 0.0)
    else:
        outside = (pos[None] > pos[:, None]) | (pos[:, None] - pos[None] >= c.window)
        leak = p[..., outside].abs().max().item()
        print(f"layer {i} sliding  rope {a.span}채널  window {c.window}")
        report(f"layer {i} window 밖·미래 확률 0", leak == 0, leak)
    report(f"layer {i} 확률 행 합 = 1", err < 1e-5, err)

### padding 무관성

원본은 요청을 이어 붙인 packed 입력이라 padding이 없습니다. 내 구현은 [B, T]에 오른쪽 padding을 두는데,
causal이라 padding이 앞쪽 출력에 영향을 주지 않아야 합니다. 묶음으로 돌린 결과와 요청 하나씩 돌린 결과를 비교합니다.

In [ ]:
from miniarch.models import step5 as S5
ref, inp = S.ref()
m = S5.verify_pair(ref)["mine"].to(dev).eval()
x, L = inp["input_ids"].to(dev), inp["seq_lens"].to(dev)
with torch.no_grad():
    full = m(x, L).logits
    for b in range(x.shape[0]):
        l = int(L[b])
        solo = m(x[b:b + 1, :l]).logits
        err = (full[b, :l] - solo[0]).abs().max().item()
        report(f"요청 {b} (길이 {l}) 단독 == 묶음", err < 1e-4, err)

### ablation 스위치

나중에 부품별 비교 실험에 쓸 스위치를 하나씩 끄고 forward가 정상(유한값)인지 확인합니다.

In [ ]:
import dataclasses
from miniarch.models import step5 as S5
ref, inp = S.ref()
base = S5.config_from_hf(ref.config)
x = inp["input_ids"].to(dev)
for sw in ["use_sparse", "use_sliding", "use_attn_gate", "use_partial_rope", "use_rope_scaling",
           "use_moe", "use_shared_expert", "use_router_bias", "use_swiglu_limit"]:
    mm = S5.Step5(dataclasses.replace(base, **{sw: False})).to(dev).eval()
    with torch.no_grad():
        out = mm(x).logits
    report(f"{sw}=False forward", bool(torch.isfinite(out).all()), 0.0)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A9_step5.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A9_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")